# nanoGPT · ROCStories
End-to-end pipeline: **data prep → model definition → training → evaluation**

## 1 · Imports

In [15]:
import os, math, time, pickle, json, re, inspect, shutil, subprocess
from contextlib import nullcontext
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed import init_process_group, destroy_process_group

import tiktoken
from datasets import load_dataset

## 2 · Data Preparation (ROCStories)

In [17]:
def story_text(example: dict) -> str:
    """Extract story text — prefers 'story', then sentence1-5, then 'text'."""
    if example.get('story'):
        return str(example['story']).strip()
    keys = [k for k in ('sentence1','sentence2','sentence3','sentence4','sentence5') if example.get(k)]
    if keys:
        return ' '.join(str(example[k]).strip() for k in keys).strip()
    if example.get('text'):
        return str(example['text']).strip()
    return ''

def tokenize_split(split, encoder) -> np.ndarray:
    """Flatten a dataset split into one uint16 token array, EOT-delimited."""
    ids = []
    for example in split:
        text = story_text(example)
        if not text:
            continue
        ids.extend(encoder.encode_ordinary(text))
        ids.append(encoder.eot_token)
    return np.array(ids, dtype=np.uint16)

# --- Download & split ---
print('Downloading ROCStories from Hugging Face …')
full_dataset = load_dataset('mintujupally/ROCStories')
encoder      = tiktoken.get_encoding('gpt2')

test_split  = full_dataset['test']
train_val   = full_dataset['train'].train_test_split(test_size=0.2, seed=1337, shuffle=True)
train_split = train_val['train']
val_split   = train_val['test']
print(f'  train: {len(train_split):,}  val: {len(val_split):,}  test: {len(test_split):,}')

# --- Tokenize & save ---
os.makedirs('data/rocstories', exist_ok=True)
for name, split in {'train': train_split, 'val': val_split}.items():
    print(f'Tokenising {name} …')
    ids = tokenize_split(split, encoder)
    ids.tofile(f'data/rocstories/{name}.bin')
    print(f'  → data/rocstories/{name}.bin  ({len(ids):,} tokens, {ids.nbytes/1e6:.1f} MB)')

# Save test split as plain text
with open('data/rocstories/test.txt', 'w', encoding='utf-8') as f:
    for example in test_split:
        text = story_text(example)
        if text:
            f.write(text + '\n\n')
print('Saved test.txt with test stories.')

Repo card metadata block was not found. Setting CardData to empty.


  train: 62,822  val: 15,706  test: 19,633
Tokenising train …
  → data/rocstories/train.bin  (3,290,634 tokens, 6.6 MB)
Tokenising val …
  → data/rocstories/val.bin  (820,507 tokens, 1.6 MB)
Saved test.txt with test stories.


## 3 · Model Definition (GPT)

In [18]:
class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias   = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class LayerRMSNorm(nn.Module):
    def __init__(self, ndim):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
    def forward(self, x):
        return F.rms_norm(x, self.weight.shape, self.weight, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn       = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj       = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout= nn.Dropout(config.dropout)
        self.n_head, self.n_embd, self.dropout = config.n_head, config.n_embd, config.dropout
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            print('WARNING: using slow attention. Flash Attention requires PyTorch >= 2.0')
            self.register_buffer('bias', torch.tril(torch.ones(config.block_size, config.block_size))
                                         .view(1, 1, config.block_size, config.block_size))
    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        if self.flash:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None,
                dropout_p=self.dropout if self.training else 0, is_causal=True)
        else:
            att = (q @ k.transpose(-2,-1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
            att = self.attn_dropout(F.softmax(att, dim=-1))
            y   = att @ v
        y = y.transpose(1,2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerRMSNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerRMSNorm(config.n_embd)
        self.mlp  = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int   = 1024
    vocab_size: int   = 50304
    n_layer:    int   = 12
    n_head:     int   = 12
    n_embd:     int   = 768
    dropout:    float = 0.0
    bias:       bool  = True

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None and config.block_size is not None
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(config.vocab_size, config.n_embd),
            wpe  = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h    = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight  # weight tying
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))
        print('number of parameters: %.2fM' % (self.get_num_params()/1e6,))

    def get_num_params(self, non_embedding=True):
        n = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n -= self.transformer.wpe.weight.numel()
        return n

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t   = idx.size()
        assert t <= self.config.block_size
        pos     = torch.arange(0, t, dtype=torch.long, device=device)
        x       = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss   = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss   = None
        return logits, loss

    def crop_block_size(self, block_size):
        assert block_size <= self.config.block_size
        self.config.block_size = block_size
        self.transformer.wpe.weight = nn.Parameter(self.transformer.wpe.weight[:block_size])
        for block in self.transformer.h:
            if hasattr(block.attn, 'bias'):
                block.attn.bias = block.attn.bias[:,:,:block_size,:block_size]

    @classmethod
    def from_pretrained(cls, model_type, override_args=None):
        assert model_type in {'gpt2','gpt2-medium','gpt2-large','gpt2-xl'}
        override_args = override_args or {}
        assert all(k == 'dropout' for k in override_args)
        from transformers import GPT2LMHeadModel
        print('loading weights from pretrained gpt: %s' % model_type)
        config_args = {
            'gpt2':        dict(n_layer=12, n_head=12, n_embd=768),
            'gpt2-medium': dict(n_layer=24, n_head=16, n_embd=1024),
            'gpt2-large':  dict(n_layer=36, n_head=20, n_embd=1280),
            'gpt2-xl':     dict(n_layer=48, n_head=25, n_embd=1600),
        }[model_type]
        config_args.update(vocab_size=50257, block_size=1024, bias=True)
        if 'dropout' in override_args:
            config_args['dropout'] = override_args['dropout']
        config = GPTConfig(**config_args)
        model  = GPT(config)
        sd     = model.state_dict()
        sd_keys = [k for k in sd if not k.endswith('.attn.bias')]
        model_hf   = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf      = model_hf.state_dict()
        sd_keys_hf = [k for k in sd_hf if not k.endswith(('.attn.masked_bias', '.attn.bias'))]
        transposed = ['attn.c_attn.weight','attn.c_proj.weight','mlp.c_fc.weight','mlp.c_proj.weight']
        assert len(sd_keys_hf) == len(sd_keys)
        for k in sd_keys_hf:
            with torch.no_grad():
                if any(k.endswith(w) for w in transposed):
                    sd[k].copy_(sd_hf[k].t())
                else:
                    sd[k].copy_(sd_hf[k])
        return model

    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
        param_dict    = {pn: p for pn, p in self.named_parameters() if p.requires_grad}
        decay_params  = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params= [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups  = [
            {'params': decay_params,   'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0},
        ]
        print(f'decayed params: {sum(p.numel() for p in decay_params):,} | '
              f'non-decayed: {sum(p.numel() for p in nodecay_params):,}')
        use_fused = 'fused' in inspect.signature(torch.optim.AdamW).parameters and device_type == 'cuda'
        optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas,
                                      **(dict(fused=True) if use_fused else {}))
        print(f'using fused AdamW: {use_fused}')
        return optimizer

    def estimate_mfu(self, fwdbwd_per_iter, dt):
        N = self.get_num_params()
        L, H, Q, T = self.config.n_layer, self.config.n_head, self.config.n_embd//self.config.n_head, self.config.block_size
        flops_per_iter = (6*N + 12*L*H*Q*T) * T * fwdbwd_per_iter
        return flops_per_iter / (dt * 312e12)  # A100 bfloat16 peak

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            idx = torch.cat((idx, torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)), dim=1)
        return idx

## 4 · Training Config (ROCStories)

In [20]:
# ── Output & checkpointing ────────────────────────────────────────────────────
out_dir               = 'out-rocstories'
init_from             = 'scratch'   # 'scratch' | 'resume'
always_save_checkpoint= False

# ── W&B logging ───────────────────────────────────────────────────────────────
wandb_log      = True
wandb_project  = 'rocstories'
wandb_run_name = 'rocstories'

# ── Dataset ───────────────────────────────────────────────────────────────────
dataset = 'rocstories'

# ── Architecture (~10M params) ────────────────────────────────────────────────
block_size = 256
n_layer    = 6
n_head     = 6
n_embd     = 384   # must be divisible by n_head
bias       = False

# ── Batch & memory ────────────────────────────────────────────────────────────
batch_size                  = 64
gradient_accumulation_steps = 16    # effective batch = 1024
dtype   = 'float32'
device  = 'cuda'
compile = True

# ── Regularization ────────────────────────────────────────────────────────────
dropout      = 0.2
weight_decay = 0.2

# ── Learning rate schedule ────────────────────────────────────────────────────
learning_rate = 3e-5
min_lr        = 3e-7
warmup_iters  = 1000
beta1, beta2  = 0.9, 0.99
decay_lr      = True

# ── Iteration & evaluation ────────────────────────────────────────────────────
max_iters      = 50000
lr_decay_iters = 50000
eval_interval  = 500
eval_iters     = 400
log_interval   = 50
eval_only      = False
grad_clip      = 1.0

## 5 · Training Loop

In [ ]:
# ── DDP setup ─────────────────────────────────────────────────────────────────
ddp = int(os.environ.get('RANK', -1)) != -1
if ddp:
    init_process_group(backend='nccl')
    ddp_rank       = int(os.environ['RANK'])
    ddp_local_rank = int(os.environ['LOCAL_RANK'])
    ddp_world_size = int(os.environ['WORLD_SIZE'])
    device         = f'cuda:{ddp_local_rank}'
    torch.cuda.set_device(device)
    master_process = ddp_rank == 0
    seed_offset    = ddp_rank
    assert gradient_accumulation_steps % ddp_world_size == 0
    gradient_accumulation_steps //= ddp_world_size
else:
    master_process, seed_offset, ddp_world_size = True, 0, 1

config = dict(dataset=dataset, block_size=block_size, n_layer=n_layer, n_head=n_head,
              n_embd=n_embd, dropout=dropout, bias=bias, learning_rate=learning_rate,
              max_iters=max_iters, batch_size=batch_size,
              gradient_accumulation_steps=gradient_accumulation_steps)

print(f'tokens per iteration: {gradient_accumulation_steps * ddp_world_size * batch_size * block_size:,}')
if master_process:
    os.makedirs(out_dir, exist_ok=True)

torch.manual_seed(1337 + seed_offset)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
device_type = 'cuda' if 'cuda' in device else 'cpu'
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

# ── Data loader ───────────────────────────────────────────────────────────────
data_dir = os.path.join('data', dataset)

def get_batch(split):
    data = np.memmap(os.path.join(data_dir, f'{split}.bin'), dtype=np.uint16, mode='r')
    ix   = torch.randint(len(data) - block_size, (batch_size,))
    x    = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    y    = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix])
    if device_type == 'cuda':
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y

# ── Model init ────────────────────────────────────────────────────────────────
iter_num, best_val_loss, best_eval_ppl = 0, 1e9, float('inf')

meta_path = os.path.join(data_dir, 'meta.pkl')
meta_vocab_size = None
if os.path.exists(meta_path):
    with open(meta_path, 'rb') as f:
        meta_vocab_size = pickle.load(f).get('vocab_size')
    print(f'found vocab_size = {meta_vocab_size}')

model_args = dict(n_layer=n_layer, n_head=n_head, n_embd=n_embd, block_size=block_size,
                  bias=bias, vocab_size=None, dropout=dropout)

if init_from == 'scratch':
    print('Initializing a new model from scratch')
    model_args['vocab_size'] = meta_vocab_size or 50304
    model = GPT(GPTConfig(**model_args))
elif init_from == 'resume':
    print(f'Resuming training from {out_dir}')
    checkpoint = torch.load(os.path.join(out_dir, 'ckpt.pt'), map_location=device)
    for k in ['n_layer','n_head','n_embd','block_size','bias','vocab_size']:
        model_args[k] = checkpoint['model_args'][k]
    model = GPT(GPTConfig(**model_args))
    state_dict = checkpoint['model']
    for k in list(state_dict):
        if k.startswith('_orig_mod.'):
            state_dict[k[10:]] = state_dict.pop(k)
    model.load_state_dict(state_dict)
    iter_num, best_val_loss = checkpoint['iter_num'], checkpoint['best_val_loss']
    best_eval_ppl = checkpoint.get('best_eval_ppl', float('inf'))
elif init_from.startswith('gpt2'):
    model = GPT.from_pretrained(init_from, dict(dropout=dropout))
    for k in ['n_layer','n_head','n_embd','block_size','bias','vocab_size']:
        model_args[k] = getattr(model.config, k)

if block_size < model.config.block_size:
    model.crop_block_size(block_size)
    model_args['block_size'] = block_size
model.to(device)

scaler    = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))
optimizer = model.configure_optimizers(weight_decay, learning_rate, (beta1, beta2), device_type)
if init_from == 'resume':
    optimizer.load_state_dict(checkpoint['optimizer'])
checkpoint = None

if compile:
    print('compiling the model…')
    model = torch.compile(model)
if ddp:
    model = DDP(model, device_ids=[ddp_local_rank])

# ── Helpers ───────────────────────────────────────────────────────────────────
@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            with ctx:
                _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

def get_lr(it):
    if it < warmup_iters:
        return learning_rate * (it + 1) / (warmup_iters + 1)
    if it > lr_decay_iters:
        return min_lr
    coeff = 0.5 * (1.0 + math.cos(math.pi * (it - warmup_iters) / (lr_decay_iters - warmup_iters)))
    return min_lr + coeff * (learning_rate - min_lr)

def run_external_eval():
    """Run eval.py subprocess and parse PPL from stdout."""
    result = subprocess.run(['python', 'eval.py', f'--out_dir={out_dir}', '--init_from=resume'],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print('eval.py failed:\n', result.stderr)
        return None
    for pattern in [r'PPL_RESULT=([0-9]+(?:\.[0-9]+)?)',
                    r'(?:ppl|perplexity)\s*[:=]?\s*([0-9]+(?:\.[0-9]+)?)']: 
        m = re.search(pattern, result.stdout, re.IGNORECASE)
        if m:
            return float(m.group(1))
    print('Could not parse PPL from eval.py output')
    return None

# ── W&B ───────────────────────────────────────────────────────────────────────
if wandb_log and master_process:
    import wandb
    wandb.init(project=wandb_project, name=wandb_run_name, config=config)

# ── Training loop ─────────────────────────────────────────────────────────────
X, Y = get_batch('train')
t0 = time.time()
local_iter_num = 0
raw_model      = model.module if ddp else model
running_mfu    = -1.0

while True:
    lr = get_lr(iter_num) if decay_lr else learning_rate
    for pg in optimizer.param_groups:
        pg['lr'] = lr

    if iter_num % eval_interval == 0 and master_process:
        losses = estimate_loss()
        print(f'step {iter_num}: train {losses["train"]:.4f}, val {losses["val"]:.4f}, '
              f'ppl {math.exp(losses["val"]):.2f}')
        if wandb_log:
            wandb.log({'iter': iter_num, 'train/loss': losses['train'],
                       'val/loss': losses['val'], 'lr': lr, 'mfu': running_mfu * 100})
        if losses['val'] < best_val_loss or always_save_checkpoint:
            best_val_loss = losses['val']
            if iter_num > 0:
                ckpt = {'model': raw_model.state_dict(), 'optimizer': optimizer.state_dict(),
                        'model_args': model_args, 'iter_num': iter_num,
                        'best_val_loss': best_val_loss, 'best_eval_ppl': best_eval_ppl, 'config': config}
                print(f'saving checkpoint to {out_dir}')
                torch.save(ckpt, os.path.join(out_dir, 'ckpt.pt'))

    if iter_num % eval_interval == 0 and master_process and iter_num > 0:
        ppl = run_external_eval()
        if ppl is not None:
            print(f'external eval ppl: {ppl:.4f}')
            if ppl < best_eval_ppl:
                best_eval_ppl = ppl
                src = os.path.join(out_dir, 'ckpt.pt')
                dst = os.path.join(out_dir, 'ckpt_best.pt')
                if os.path.exists(src):
                    shutil.copyfile(src, dst)
                    print(f'New best ppl {best_eval_ppl:.4f} → {dst}')

    if iter_num == 0 and eval_only:
        break

    for micro_step in range(gradient_accumulation_steps):
        if ddp:
            model.require_backward_grad_sync = (micro_step == gradient_accumulation_steps - 1)
        with ctx:
            _, loss = model(X, Y)
            loss = loss / gradient_accumulation_steps
        X, Y = get_batch('train')
        scaler.scale(loss).backward()

    if grad_clip != 0.0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    dt = time.time() - t0; t0 = time.time()
    if iter_num % log_interval == 0 and master_process:
        lossf = loss.item() * gradient_accumulation_steps
        if local_iter_num >= 5:
            running_mfu = raw_model.estimate_mfu(batch_size * gradient_accumulation_steps, dt)
        print(f'iter {iter_num}: loss {lossf:.4f}, time {dt*1000:.2f}ms, mfu {running_mfu*100:.2f}%')

    iter_num += 1; local_iter_num += 1
    if iter_num > max_iters:
        break

if ddp:
    destroy_process_group()

tokens per iteration: 262,144
Initializing a new model from scratch
number of parameters: 29.94M


/tmp/ipykernel_17343/3683220215.py:87: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))


decayed params: 30,031,872 | non-decayed: 4,992
using fused AdamW: True
compiling the model…


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hadisssurya (hadissurya) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


step 0: train 10.9176, val 10.9174, ppl 55128.20
iter 0: loss 10.9114, time 35616.94ms, mfu -100.00%
iter 50: loss 10.5547, time 832.01ms, mfu 18.85%
iter 100: loss 9.9545, time 832.59ms, mfu 18.84%
iter 150: loss 9.7029, time 833.06ms, mfu 18.83%
iter 200: loss 9.4260, time 833.35ms, mfu 18.82%
iter 250: loss 9.1739, time 832.27ms, mfu 18.85%
iter 300: loss 8.8755, time 833.53ms, mfu 18.82%
iter 350: loss 8.5444, time 833.55ms, mfu 18.82%
iter 400: loss 8.2085, time 833.26ms, mfu 18.83%
iter 450: loss 7.8624, time 833.21ms, mfu 18.83%
step 500: train 7.4545, val 7.4545, ppl 1727.70
saving checkpoint to out-rocstories
eval.py failed:
 python3: can't open file '/content/eval.py': [Errno 2] No such file or directory

iter 500: loss 7.5623, time 16241.67ms, mfu 0.97%
iter 550: loss 7.2054, time 831.15ms, mfu 18.87%
iter 600: loss 6.8763, time 830.74ms, mfu 18.88%
iter 650: loss 6.5783, time 832.38ms, mfu 18.85%
iter 700: loss 6.2398, time 831.30ms, mfu 18.87%
iter 750: loss 6.0457, time 8

## 6 · Evaluation

In [ ]:
# ── Eval config ───────────────────────────────────────────────────────────────
eval_init_from  = 'resume'                    # 'resume' or 'gpt2', 'gpt2-medium', …
eval_out_dir    = 'out-rocstories'
eval_device     = 'cuda'
eval_dtype      = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
eval_compile    = False
eval_seed       = 1337

input_file      = 'data/rocstories/test.txt'
input_format    = 'txt'   # 'txt' | 'jsonl' | 'json' | 'auto'
json_text_key   = 'text'
max_paragraphs  = -1      # -1 = all
print_first_n   = 3

# ── Paragraph loaders ─────────────────────────────────────────────────────────
def _read_txt_paragraphs(path):
    return [p.strip() for p in open(path, encoding='utf-8').read().split('\n\n') if p.strip()]

def _read_jsonl_paragraphs(path, key):
    out = []
    for ln, line in enumerate(open(path, encoding='utf-8'), 1):
        if not line.strip(): continue
        obj = json.loads(line)
        text = obj if isinstance(obj, str) else obj[key]
        if text.strip(): out.append(text.strip())
    return out

def _read_json_paragraphs(path, key):
    data = json.load(open(path, encoding='utf-8'))
    assert isinstance(data, list)
    return [((item if isinstance(item, str) else item[key])).strip() for item in data
            if (item if isinstance(item, str) else item.get(key, '')).strip()]

def load_paragraphs(path, fmt, key):
    if fmt == 'auto':
        fmt = {'txt': 'txt', 'jsonl': 'jsonl', 'json': 'json'}.get(os.path.splitext(path)[1][1:], 'txt')
    fn  = {'txt': _read_txt_paragraphs, 'jsonl': _read_jsonl_paragraphs, 'json': _read_json_paragraphs}[fmt]
    return (fn(path) if fmt == 'txt' else fn(path, key)), fmt

# ── Setup ─────────────────────────────────────────────────────────────────────
torch.manual_seed(eval_seed)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
eval_device_type = 'cuda' if 'cuda' in eval_device else 'cpu'
eval_ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[eval_dtype]
eval_ctx = nullcontext() if eval_device_type == 'cpu' else torch.amp.autocast(device_type=eval_device_type, dtype=eval_ptdtype)

# ── Load model ────────────────────────────────────────────────────────────────
if eval_init_from == 'resume':
    checkpoint  = torch.load(os.path.join(eval_out_dir, 'ckpt.pt'), map_location=eval_device)
    eval_model  = GPT(GPTConfig(**checkpoint['model_args']))
    state_dict  = checkpoint['model']
    for k in list(state_dict):
        if k.startswith('_orig_mod.'): state_dict[k[10:]] = state_dict.pop(k)
    eval_model.load_state_dict(state_dict)
elif eval_init_from.startswith('gpt2'):
    eval_model = GPT.from_pretrained(eval_init_from, dict(dropout=0.0))

eval_model.eval().to(eval_device)
if eval_compile:
    eval_model = torch.compile(eval_model)

enc    = tiktoken.get_encoding('gpt2')
encode = lambda s: enc.encode(s, allowed_special={'<|endoftext|>'})

# ── Load & preview paragraphs ─────────────────────────────────────────────────
paragraphs, used_fmt = load_paragraphs(input_file, input_format, json_text_key)
if max_paragraphs >= 0:
    paragraphs = paragraphs[:max_paragraphs]
assert paragraphs, f'No paragraphs found in {input_file}'
print(f'Loaded {len(paragraphs)} paragraphs (format={used_fmt})')
for i, p in enumerate(paragraphs[:print_first_n]):
    print(f'[preview {i}] {p.replace(chr(10)," ")[:120]}')

# ── Evaluate ──────────────────────────────────────────────────────────────────
total_nll, total_tokens, used_paragraphs, skipped_short = 0.0, 0, 0, 0
eval_block_size = eval_model.config.block_size

with torch.no_grad(), eval_ctx:
    for para in paragraphs:
        token_ids = encode(para)
        if len(token_ids) < 2: skipped_short += 1; continue
        pos = 0
        while pos < len(token_ids) - 1:
            inp = token_ids[pos: pos + eval_block_size]
            tgt = token_ids[pos + 1: pos + 1 + eval_block_size]
            if not tgt: break
            inp = inp[:len(tgt)]
            x = torch.tensor(inp, dtype=torch.long, device=eval_device)[None, :]
            y = torch.tensor(tgt, dtype=torch.long, device=eval_device)[None, :]
            _, loss = eval_model(x, y)
            total_nll    += loss.item() * len(tgt)
            total_tokens += len(tgt)
            pos          += len(tgt)
        used_paragraphs += 1

assert total_tokens > 0, 'No valid tokens to evaluate.'
avg_loss = total_nll / total_tokens
ppl      = math.exp(avg_loss)

print('----- Evaluation Results -----')
print(f'model           : {eval_init_from}')
print(f'paragraphs_used : {used_paragraphs}')
print(f'paragraphs_skip : {skipped_short}')
print(f'pred_tokens     : {total_tokens}')
print(f'avg_loss        : {avg_loss:.3f}')
print(f'ppl             : {ppl:.2f}')